# 4.2 Dataset Statistics
**Children's Stories Fine-Tuning — NLP Mini Project**

This notebook reproduces the dataset analysis for Section 4.2 of the report.
It subsamples 10,000 stories from `ajibawa-2023/Children-Stories-Collection`,
assigns synthetic age-level labels (this time using the smarter gemma 4 approach, and
produces Figure 3: distribution of story lengths and age-level categories.

# 1. INSTALL DEPENDENCIES
!pip install -q llama-cpp-python datasets pandas matplotlib seaborn huggingface_hub

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from llama_cpp import Llama
from huggingface_hub import hf_hub_download

# --- CONFIG ---
MODEL_REPO = "google/gemma-2-2b-it-GGUF" # Official GGUF repo
MODEL_FILE = "gemma-2-2b-it-Q4_K_M.gguf"
DATASET_ID = "ajibawa-2023/Children-Stories-Collection"
SAMPLE_SIZE = 10000 
# --------------

# 2. DOWNLOAD MODEL
print("Downloading Gemma-2-2B-IT GGUF...")
model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE)

# 3. INITIALIZE MODEL (GPU Accelerated)
llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1, # Offload all layers to T4 GPU
    n_ctx=2048,      # Context window
    verbose=False
)

# 4. LOAD DATASET
ds = load_dataset(DATASET_ID, split='train').shuffle(seed=42).select(range(SAMPLE_SIZE))

# 5. LABELING LOGIC
AGE_GROUPS = ["Age 3-4", "Age 5-6", "Age 7-8", "Age 9-10", "Age 11-12"]

def get_label(prompt, text):
    # Construct a classification prompt
    sys_prompt = f"Classify this children's story into one of these groups: {', '.join(AGE_GROUPS)}. Reply ONLY with the group name."
    full_prompt = f"<start_of_turn>user\n{sys_prompt}\n\nSTORY: {text[:800]}\nLabel:<end_of_turn>\n<start_of_turn>model\n"
    
    output = llm(full_prompt, max_tokens=10, stop=["<end_of_turn>"], echo=False)
    label = output['choices'][0]['text'].strip()
    
    # Validation: fallback to first group if model hallucinates
    for group in AGE_GROUPS:
        if group.lower() in label.lower():
            return group
    return "Uncategorized"

print(f"Labeling {SAMPLE_SIZE} samples (this will take a while)...")
labeled_data = []
for i, ex in enumerate(ds):
    label = get_label(ex['prompt'], ex['text'])
    labeled_data.append({
        'prompt': ex['prompt'],
        'text': ex['text'],
        'label': label,
        'word_count': len(ex['text'].split())
    })
    if (i+1) % 100 == 0: print(f"Progress: {i+1}/{SAMPLE_SIZE}")

df = pd.DataFrame(labeled_data)
df.to_csv("labeled_children_stories.csv", index=False)

# 6. VISUALIZE CHARACTERISTICS
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot A: Label Distribution (No longer uniform!)
sns.countplot(data=df, x='label', order=AGE_GROUPS, ax=axes[0], palette="viridis")
axes[0].set_title("Distribution of Model-Assigned Age Groups")
axes[0].tick_params(axis='x', rotation=45)

# Plot B: Story Length by Age Group
sns.boxplot(data=df, x='label', y='word_count', order=AGE_GROUPS, ax=axes[1], palette="magma")
axes[1].set_title("Story Length (Words) per Age Group")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("Summary Stats:")
print(df.groupby('label')['word_count'].describe())